In [ ]:
# 第7回講義資料の録音を取得し、そこから学習用データセットの作成と、モデルのファインチューニングまで行いたい。
# このファイルではスプレッドシート作成までを行う
# https://zenn.dev/k_sone/articles/4d137d58dd06a6 を参考にした

In [ ]:
from google.colab import drive
import os
import shutil

mountpoint = '/content/drive'

# Mount point に不要なファイルがある場合、ディレクトリごと削除する
if os.path.exists(mountpoint):
    if os.listdir(mountpoint):  # 中身があるか確認
        print(f"Mount point {mountpoint} is not empty. Removing entire directory...")
        shutil.rmtree(mountpoint)  # 安全に削除

# 再マウント
drive.mount(mountpoint)


Mount point /content/drive is not empty. Removing entire directory...


OSError: [Errno 125] Operation canceled: '.shortcut-targets-by-id'

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pydub

In [3]:
from pydub import AudioSegment
from pydub.silence import split_on_silence
import math
import librosa

In [4]:
source = AudioSegment.from_file("/content/drive/MyDrive/audio_model/aud/7_aud.m4a", "m4a")
audio_input, _ = librosa.load("/content/drive/MyDrive/audio_model/aud/7_aud.m4a", sr=16000)

/tmp/ipython-input-4-2309978803.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_input, _ = librosa.load("/content/drive/MyDrive/audio_model/aud/7_aud.m4a", sr=16000)
/usr/local/lib/python3.11/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [5]:
import torch
import torchaudio
model_id = "kotoba-tech/kotoba-whisper-v2.0"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
!pip install -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 18.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 881.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 15.2 MB/

In [7]:
import whisper
from datasets import Dataset,Audio
from whisper.audio import N_FRAMES, pad_or_trim, log_mel_spectrogram
from whisper.tokenizer import get_tokenizer

In [8]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(model_id).to(device)
model.config.attn_implementation = "flash_attention_2"

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.51G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

In [9]:
def split_aud(fi,chunk_sec=30):
    filelist = []
    total_chunks = math.ceil(len(fi) / 30000)
    print(total_chunks)
    for i in range(total_chunks):
        chunk = fi[i * 30000:(i + 1) * 30000]
        path = f"{outputDirname}/chunk{i}.wav"
        chunk.export(path, format="wav")
        filelist.append(path)
    return filelist


def get_string(chunk):
    input_features = processor(chunk, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    # 推論の実行
    generated_ids = model.generate(input_features)
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)
    return transcription[0]

In [10]:
import os
import shutil
outputDirname = "/content/drive/MyDrive/audio_model/output_data"
if os.path.exists(outputDirname):
  shutil.rmtree(outputDirname)

os.makedirs(outputDirname, exist_ok=True)


if len(source) / 1000 > 30:  # sourceはms単位の長さ
    chunk_files = split_aud(source)
    print(chunk_files)

10
['/content/drive/MyDrive/audio_model/output_data/chunk0.wav', '/content/drive/MyDrive/audio_model/output_data/chunk1.wav', '/content/drive/MyDrive/audio_model/output_data/chunk2.wav', '/content/drive/MyDrive/audio_model/output_data/chunk3.wav', '/content/drive/MyDrive/audio_model/output_data/chunk4.wav', '/content/drive/MyDrive/audio_model/output_data/chunk5.wav', '/content/drive/MyDrive/audio_model/output_data/chunk6.wav', '/content/drive/MyDrive/audio_model/output_data/chunk7.wav', '/content/drive/MyDrive/audio_model/output_data/chunk8.wav', '/content/drive/MyDrive/audio_model/output_data/chunk9.wav']


In [14]:
datasets = Dataset.from_dict({"audio": chunk_files}).cast_column("audio", Audio(sampling_rate=16000))

In [15]:
datasets[0]["audio"]

{'path': '/content/drive/MyDrive/audio_model/output_data/chunk0.wav',
 'array': array([0.        , 0.        , 0.        , ..., 0.00916471, 0.01024066,
        0.01280076]),
 'sampling_rate': 16000}

In [17]:
predict_data = []
for i in range(len(chunk_files)):
  audio_input, _ = librosa.load(chunk_files[i], sr=16000)
  predict_data.append({**datasets[i], "sentence":get_string(audio_input)})
  print(f"チャンク{i}番目完了")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


チャンク0番目完了
チャンク1番目完了
チャンク2番目完了
チャンク3番目完了
チャンク4番目完了
チャンク5番目完了
チャンク6番目完了
チャンク7番目完了
チャンク8番目完了
チャンク9番目完了


In [21]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

from subprocess import getoutput
!apt-get install xattr > /dev/null

spread_sheet_dir_id = getoutput("xattr -p 'user.drive.id' " + "'" + outputDirname[:-11] + "'")
ss = gc.create('学習データ', folder_id=spread_sheet_dir_id)
#ss.append_row( values= ['url', 'colab_path','sampling_rate','correct','whisper'] )
ss.values_append("シート1", {"valueInputOption": "USER_ENTERED"}, {"values": list(map(
    lambda out: [
        f"""https://drive.google.com/file/d/{getoutput("xattr -p 'user.drive.id' " + "'" + out["audio"]["path"] + "'")}""",
        out["audio"]["path"],
        out["audio"]["sampling_rate"],
        "",
        out["sentence"]
    ], predict_data))})


{'spreadsheetId': '1WZdCKCovlYurI-DkcAMI23LKQyaiyFTIMiDvICNP_VI',
 'updates': {'spreadsheetId': '1WZdCKCovlYurI-DkcAMI23LKQyaiyFTIMiDvICNP_VI',
  'updatedRange': "'シート1'!A1:E10",
  'updatedRows': 10,
  'updatedColumns': 5,
  'updatedCells': 50}}